# 00 - Dataset Ingestion and Preparation

This notebook manages the initial setup, ingestion, validation, and export of the **Corr2Cause** dataset for the causal NLP reasoning pipeline.

### Sections
1. **Project Setup**: Initialize directory paths and configure reusable export and validation logic.
2. **Load Corr2Cause Dataset**: Ingest the benchmark from Hugging Face Hub.
3. **Inspect Dataset Splits**: Examine split record counts and partition structure.
4. **Inspect Dataset Schema and Samples**: Analyze feature definitions, missing values, duplicates, and sample records.
5. **Analyze Label Distribution**: Evaluate class balance, variable counts, and causal template frequencies.
6. **Export Original Splits**: Perform preflight conflict checks and export/reuse raw splits.
7. **Combine Training and Validation**: Concatenate training and validation sets for downstream training.
8. **Export Combined Dataset**: Export/reuse combined data and separate test partition in `data/processed/`.
9. **Verify Exported Datasets**: Audit all exported CSV files for schema, row counts, and data integrity across all columns.
10. **Summary**: Synthesize dataset characteristics, split isolation, and pipeline next steps.

## 1. Project Setup

Initialize repository-relative directory paths for raw and processed datasets, define target export paths, and configure reusable data verification and safe export routines.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import pandas as pd
from datasets import load_dataset

def find_repo_root() -> Path:
    """Locate the repository root whether run from root, AML/, DL/, or subfolders."""
    current = Path.cwd().resolve()
    for candidate in [current, current.parent, current.parent.parent]:
        if (candidate / "00_setup.ipynb").exists() and (candidate / "data").exists():
            return candidate
        if (candidate / ".git").exists() and (candidate / "AML").exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate repository root from cwd: '{current}'. "
        "Ensure notebook resides within the causal-nlp-reasoning repository."
    )

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Standardized output file paths across pipeline
EXPECTED_EXPORT_PATHS = {
    "raw_train": RAW_DATA_DIR / "train.csv",
    "raw_val": RAW_DATA_DIR / "validation.csv",
    "raw_test": RAW_DATA_DIR / "test.csv",
    "processed_train_val": PROCESSED_DATA_DIR / "train_val.csv",
    "processed_test": PROCESSED_DATA_DIR / "test.csv",
}

# -----------------------------------------------------------------------------
# Configuration: Normal Setup vs. Fresh Rebuild Workflow
# -----------------------------------------------------------------------------
# FORCE_REBUILD:
#   False (normal setup): Safe, non-destructive mode. Reuses valid, pre-existing
#          exported files without rewriting.
#   True  (fresh data rebuild): Explicitly regenerates the five canonical CSV
#          files from Hugging Face: data/raw/{train,validation,test}.csv and
#          data/processed/{train_val,test}.csv. Never touches other directories.
FORCE_REBUILD = False

print(f"Project root             : {REPO_ROOT.name}")
print(f"Raw data directory       : {RAW_DATA_DIR.relative_to(REPO_ROOT)}")
print(f"Processed data directory : {PROCESSED_DATA_DIR.relative_to(REPO_ROOT)}")
print(f"Rebuild workflow mode    : {'FRESH REBUILD (FORCE_REBUILD=True)' if FORCE_REBUILD else 'SAFE REUSE'}")
print("Configured 5 expected destinations with safe export and verification routines.")


Project root             : causal-nlp-reasoning
Raw data directory       : data/raw
Processed data directory : data/processed
Rebuild workflow mode    : SAFE REUSE
Configured 5 expected destinations with safe export and verification routines.


### Safe Export and Verification Helpers
Helper routines to verify schema fidelity and prevent accidental overwrites.

In [2]:
def verify_dataframe_match(df_loaded, df_expected, dataset_label=""):
    """Verify df_loaded matches df_expected across columns, count, and values.
    Returns (True, None) if match, or (False, reason_str) if mismatch.
    """
    if len(df_loaded) != len(df_expected):
        return False, f"Row count mismatch: found {len(df_loaded):,}, expected {len(df_expected):,}"
    
    if list(df_loaded.columns) != list(df_expected.columns):
        return False, f"Column mismatch: found {list(df_loaded.columns)}, expected {list(df_expected.columns)}"
    
    for col in df_expected.columns:
        s_expected = df_expected[col].fillna("").astype(str).reset_index(drop=True)
        s_loaded = df_loaded[col].fillna("").astype(str).reset_index(drop=True)
        if not s_loaded.equals(s_expected):
            diff_indices = (s_loaded != s_expected)
            first_idx = int(diff_indices.idxmax())
            return False, (
                f"Value mismatch in column '{col}' at record index {first_idx}: "
                f"found '{s_loaded[first_idx][:40]}...', expected '{s_expected[first_idx][:40]}...'"
            )
            
    return True, None


### Preflight Verification Across Destinations
Inspect destination paths prior to export.


In [3]:
def preflight_check_all_destinations(destinations_dict, force_rebuild=False):
    """Inspect all expected output destinations before any writing takes place.
    If force_rebuild is False and any existing file contains conflicting data, halts
    before any missing file is written.
    """
    conflicts = []
    preflight_records = []
    for name, (path, df_expected) in destinations_dict.items():
        rel_path = str(path.relative_to(REPO_ROOT))
        if path.exists():
            df_existing = pd.read_csv(path)
            is_match, reason = verify_dataframe_match(df_existing, df_expected, dataset_label=name)
            if force_rebuild:
                status = "Will overwrite (Fresh Rebuild)"
                preflight_records.append({"Dataset": name, "Path": rel_path, "Status": status, "Rows": len(df_existing), "Detail": "Target marked for fresh export"})
            elif is_match:
                status = "Already exists and verified"
                preflight_records.append({"Dataset": name, "Path": rel_path, "Status": status, "Rows": len(df_existing), "Detail": "Matches expected dataset"})
            else:
                status = "Conflict detected"
                conflicts.append((rel_path, reason))
                preflight_records.append({"Dataset": name, "Path": rel_path, "Status": status, "Rows": len(df_existing), "Detail": reason})
        else:
            status = "Missing (ready to export)"
            preflight_records.append({"Dataset": name, "Path": rel_path, "Status": status, "Rows": len(df_expected), "Detail": "File does not exist yet"})

    if conflicts and not force_rebuild:
        conflict_msg = "\n".join(f"  - {p}: {err}" for p, err in conflicts)
        raise ValueError(
            f"Preflight validation failed: Existing file(s) contain conflicting data:\n{conflict_msg}\n"
            "Execution halted before writing any files. Conflicting files are preserved without modification."
        )
    return pd.DataFrame(preflight_records)


### Safe Export and Rebuild Handler
Provides idempotent exports with explicit `force_rebuild` support.


In [4]:
def export_dataset(path, df_expected, dataset_name="Dataset", force_rebuild=False):
    """Safely exports missing datasets, reuses verified existing datasets, or
    executes an explicit fresh rebuild when force_rebuild=True.
    
    Safety constraints:
    - Never deletes directories or un-targeted files.
    - If deletion is needed for force_rebuild, only targets the exact 5 generated files.
    - Re-reads and strictly verifies exported file contents before returning.
    """
    path = Path(path)
    rel_path = str(path.relative_to(REPO_ROOT))
    allowed_targets = set(EXPECTED_EXPORT_PATHS.values())

    if path.exists() and not force_rebuild:
        df_loaded = pd.read_csv(path)
        is_match, reason = verify_dataframe_match(df_loaded, df_expected, dataset_label=dataset_name)
        if is_match:
            return {
                "Dataset": dataset_name,
                "Path": rel_path,
                "Rows": len(df_loaded),
                "Action": "Verified and Reused (Matches existing)",
            }
        else:
            raise ValueError(
                f"Conflict detected for '{dataset_name}' at {rel_path}: {reason}. "
                "Will not overwrite existing file."
            )

    if path.exists() and force_rebuild:
        if path not in allowed_targets:
            raise PermissionError(f"Safety restriction: Refusing to delete non-target file: {rel_path}")
        path.unlink()

    path.parent.mkdir(parents=True, exist_ok=True)
    df_expected.to_csv(path, index=False)

    df_written = pd.read_csv(path)
    is_match, reason = verify_dataframe_match(df_written, df_expected, dataset_label=dataset_name)
    if not is_match:
        raise ValueError(f"Post-write validation failed for {rel_path}: {reason}")

    return {
        "Dataset": dataset_name,
        "Path": rel_path,
        "Rows": len(df_written),
        "Action": "Exported fresh dataset and verified",
    }

# Backward compatibility alias
export_or_reuse_dataset = export_dataset


## 2. Load Corr2Cause Dataset

The **Corr2Cause** benchmark (Jin et al., 2023) evaluates natural language processing models on causal reasoning from correlational statements.

The task is framed as natural language inference:
- **Premise**: Textual description of statistical relations (correlations and conditional independencies) in a closed system of variables.
- **Hypothesis**: A candidate causal relationship between specific variables.
- **Label**: Binary target indicating whether the causal hypothesis is logically entailed (`1`) or refuted (`0`) by the premise.

In [5]:
DATASET_NAME = "causal-nlp/corr2cause"
print(f"Loading '{DATASET_NAME}' from Hugging Face...")
dataset = load_dataset(DATASET_NAME)
print("Dataset successfully loaded.")
print(dataset)

Loading 'causal-nlp/corr2cause' from Hugging Face...


Dataset successfully loaded.
DatasetDict({
    train: Dataset({
        features: ['input', 'label', 'num_variables', 'template'],
        num_rows: 205734
    })
    test: Dataset({
        features: ['input', 'label', 'num_variables', 'template'],
        num_rows: 1162
    })
    validation: Dataset({
        features: ['input', 'label', 'num_variables', 'template'],
        num_rows: 1076
    })
})


## 3. Inspect Dataset Splits

Verify the presence of required dataset partitions (`train`, `validation`, and `test`) and evaluate row counts across splits.

In [6]:
REQUIRED_SPLITS = {"train", "validation", "test"}
available_splits = set(dataset.keys())

missing_splits = REQUIRED_SPLITS - available_splits
if missing_splits:
    raise ValueError(f"Required dataset split(s) missing from loaded dataset: {missing_splits}")

splits_table = []
for split_name in ["train", "validation", "test"]:
    split_ds = dataset[split_name]
    splits_table.append({
        "Split": split_name,
        "Record Count": len(split_ds),
        "Features": ", ".join(split_ds.features.keys()),
        "Num Features": len(split_ds.features),
    })

df_splits_summary = pd.DataFrame(splits_table)
display(df_splits_summary)
# Explicit dataset schema and row-count validation
EXPECTED_ROW_COUNTS = {
    "train": 205734,
    "validation": 1076,
    "test": 1162,
}
EXPECTED_COLUMNS = {"input", "label", "num_variables", "template"}

for split_name, expected_count in EXPECTED_ROW_COUNTS.items():
    actual_count = len(dataset[split_name])
    if actual_count != expected_count:
        raise ValueError(
            f"Row count mismatch for '{split_name}' split: expected {expected_count:,}, got {actual_count:,}"
        )
    actual_cols = set(dataset[split_name].features.keys())
    if actual_cols != EXPECTED_COLUMNS:
        raise ValueError(
            f"Column schema mismatch for '{split_name}' split: expected {EXPECTED_COLUMNS}, got {actual_cols}"
        )

print("✓ Hugging Face dataset split row counts and schemas explicitly validated.")


,Split,Record Count,Features,Num Features
0,train,205734,"input, label, num_variables, template",4
1,validation,1076,"input, label, num_variables, template",4
2,test,1162,"input, label, num_variables, template",4


✓ Hugging Face dataset split row counts and schemas explicitly validated.


## 4. Inspect Dataset Schema and Samples

Confirm the schema structure from the loaded dataset object, inspect data types, verify missing values and duplicate rows, and inspect sample records.

The expected schema comprises:
- `input`: Formatted premise and hypothesis text.
- `label`: Binary classification label (`0` or `1`).
- `num_variables`: Number of variables involved in the causal system (4, 5, or 6).
- `template`: Semantic causal reasoning relation (e.g., direct cause, collider, confounder, mediator).

In [7]:
# Extract column types directly from dataset features
print("Verified Dataset Schema:")
for col_name, feature_def in dataset["train"].features.items():
    print(f"  • {col_name:<15} : {feature_def}")

# Convert Hugging Face splits to pandas DataFrames
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()
df_test = dataset["test"].to_pandas()

# Check missing values and duplicate records across splits
quality_report = []
for name, df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    quality_report.append({
        "Split": name,
        "Total Rows": len(df),
        "Missing Values": int(df.isnull().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
    })

display(pd.DataFrame(quality_report))

Verified Dataset Schema:
  • input           : Value('large_string')
  • label           : Value('int64')
  • num_variables   : Value('int64')
  • template        : Value('large_string')


,Split,Total Rows,Missing Values,Duplicate Rows
0,Train,205734,0,0
1,Validation,1076,0,0
2,Test,1162,0,0


In [8]:
print("=" * 80)
print("SAMPLE PREMISE AND HYPOTHESIS PAIR (Train Split, Index 0)")
print("=" * 80)
sample_record = df_train.iloc[0]
print(f"Template      : {sample_record.get('template', 'N/A')}")
print(f"Num Variables : {sample_record.get('num_variables', 'N/A')}")
print(f"Target Label  : {sample_record['label']} ({'Entailment (1)' if sample_record['label'] == 1 else 'Refuted (0)'})")
print(f"\nInput Text:\n{sample_record['input']}")
print("=" * 80)

print("\nSample Records (Top 2 from Train):")
display(df_train.head(2))

SAMPLE PREMISE AND HYPOTHESIS PAIR (Train Split, Index 0)
Template      : has_collider
Num Variables : 4
Target Label  : 0 (Refuted (0))

Input Text:
Premise: Suppose there is a closed system of 4 variables, A, B, C and D. All the statistical relations among these 4 variables are as follows: A correlates with B. A correlates with C. A correlates with D. B correlates with C. B correlates with D. C correlates with D. However, B and D are independent given A. B and D are independent given A and C. C and D are independent given A. C and D are independent given A and B.
Hypothesis: There exists at least one collider (i.e., common effect) of A and B.

Sample Records (Top 2 from Train):


,input,label,num_variables,template
0,Premise: Suppose there is a closed system of 4...,0,4,has_collider
1,Premise: Suppose there is a closed system of 4...,0,4,non-child descendant


## 5. Analyze Label Distribution

Assess target class distribution across splits, as well as the complexity breakdown by variable count and causal template.

In [9]:
label_distribution = []
for name, df in [("Train", df_train), ("Validation", df_val), ("Test", df_test)]:
    counts = df["label"].value_counts().to_dict()
    props = df["label"].value_counts(normalize=True).to_dict()
    label_distribution.append({
        "Split": name,
        "Total Rows": len(df),
        "Label 0 (Refuted)": counts.get(0, 0),
        "Label 0 (%)": round(props.get(0, 0.0) * 100, 2),
        "Label 1 (Entailed)": counts.get(1, 0),
        "Label 1 (%)": round(props.get(1, 0.0) * 100, 2),
    })

display(pd.DataFrame(label_distribution))

,Split,Total Rows,Label 0 (Refuted),Label 0 (%),Label 1 (Entailed),Label 1 (%)
0,Train,205734,167686,81.51,38048,18.49
1,Validation,1076,937,87.08,139,12.92
2,Test,1162,982,84.51,180,15.49


In [10]:
if "num_variables" in df_train.columns:
    print("Variable Count Distribution (Train Split):")
    display(df_train["num_variables"].value_counts().sort_index().to_frame("Count"))

if "template" in df_train.columns:
    print("\nTop 10 Causal Templates (Train Split):")
    display(df_train["template"].value_counts().head(10).to_frame("Count"))

Variable Count Distribution (Train Split):


,Count
num_variables,
4,576
5,7524
6,197634



Top 10 Causal Templates (Train Split):


,Count
template,
non-parent ancestor,34356
parent,34292
child,34291
has_collider,34284
has_confounder,34284
non-child descendant,34227


## 6. Export Original Splits

Persist original splits to `data/raw/` (`train.csv`, `validation.csv`, `test.csv`).

### Preflight Inspection and No-Overwrite Guarantees
Before exporting any files, the pipeline inspects all five expected destinations. For any file that already exists, it verifies that the file contents match the expected dataset. If valid, the existing file is reused without rewriting. If any conflict is detected across any destination, export stops immediately before writing any files.

In [11]:
# Prepare combined dataset reference for preflight destination check
df_combined_ref = pd.concat([df_train, df_val], ignore_index=True)

all_destinations = {
    "Raw Train": (EXPECTED_EXPORT_PATHS["raw_train"], df_train),
    "Raw Validation": (EXPECTED_EXPORT_PATHS["raw_val"], df_val),
    "Raw Test": (EXPECTED_EXPORT_PATHS["raw_test"], df_test),
    "Processed Combined": (EXPECTED_EXPORT_PATHS["processed_train_val"], df_combined_ref),
    "Processed Test": (EXPECTED_EXPORT_PATHS["processed_test"], df_test),
}

# Run preflight validation across all five destinations
df_preflight = preflight_check_all_destinations(all_destinations, force_rebuild=FORCE_REBUILD)
print("Preflight Destination Inspection:")
display(df_preflight)

# Safely export or rebuild raw splits
raw_export_reports = [
    export_dataset(EXPECTED_EXPORT_PATHS["raw_train"], df_train, "Raw Train", force_rebuild=FORCE_REBUILD),
    export_dataset(EXPECTED_EXPORT_PATHS["raw_val"], df_val, "Raw Validation", force_rebuild=FORCE_REBUILD),
    export_dataset(EXPECTED_EXPORT_PATHS["raw_test"], df_test, "Raw Test", force_rebuild=FORCE_REBUILD),
]

display(pd.DataFrame(raw_export_reports))


Preflight Destination Inspection:


,Dataset,Path,Status,Rows,Detail
0,Raw Train,data/raw/train.csv,Missing (ready to export),205734,File does not exist yet
1,Raw Validation,data/raw/validation.csv,Missing (ready to export),1076,File does not exist yet
2,Raw Test,data/raw/test.csv,Missing (ready to export),1162,File does not exist yet
3,Processed Combined,data/processed/train_val.csv,Missing (ready to export),206810,File does not exist yet
4,Processed Test,data/processed/test.csv,Missing (ready to export),1162,File does not exist yet


,Dataset,Path,Rows,Action
0,Raw Train,data/raw/train.csv,205734,Exported fresh dataset and verified
1,Raw Validation,data/raw/validation.csv,1076,Exported fresh dataset and verified
2,Raw Test,data/raw/test.csv,1162,Exported fresh dataset and verified


## 7. Combine Training and Validation

Concatenate the original training (`205,734` rows) and validation (`1,076` rows) splits into a single combined dataset (`206,810` rows).

**Methodological Rationale**:
- **Model Development**: Training and validation sets remain strictly isolated during initial exploration, architecture selection, and hyperparameter tuning so validation performance accurately informs modeling decisions.
- **Final Retraining**: After optimal model architecture and hyperparameters are finalized, retraining on the combined training and validation data maximizes exposure to the full training distribution.
- **Test Set Isolation**: The test set is held out strictly for final evaluation and must never be included in the combined training corpus or used for hyperparameter selection.

In [12]:
# Downstream processed files are strictly regenerated from the freshly exported raw splits
df_raw_train_fresh = pd.read_csv(EXPECTED_EXPORT_PATHS["raw_train"])
df_raw_val_fresh = pd.read_csv(EXPECTED_EXPORT_PATHS["raw_val"])
df_combined = pd.concat([df_raw_train_fresh, df_raw_val_fresh], ignore_index=True)

# Verify row count equality
expected_combined_count = len(df_raw_train_fresh) + len(df_raw_val_fresh)
assert len(df_combined) == expected_combined_count, (
    f"Combined row count mismatch: expected {expected_combined_count}, got {len(df_combined)}"
)
assert list(df_combined.columns) == list(df_raw_train_fresh.columns), "Column schema mismatch in combined DataFrame"

print(f"Fresh Raw Training rows   : {len(df_raw_train_fresh):,}")
print(f"Fresh Raw Validation rows : {len(df_raw_val_fresh):,}")
print(f"Combined total rows       : {len(df_combined):,}")
print("Row count assertion passed: len(combined) == len(train) + len(val)")

display(df_combined.head(2))


Fresh Raw Training rows   : 205,734
Fresh Raw Validation rows : 1,076
Combined total rows       : 206,810
Row count assertion passed: len(combined) == len(train) + len(val)


,input,label,num_variables,template
0,Premise: Suppose there is a closed system of 4...,0,4,has_collider
1,Premise: Suppose there is a closed system of 4...,0,4,non-child descendant


## 8. Export Combined Dataset

Export the combined training and validation dataset to `data/processed/train_val.csv`.

Additionally, export an exact copy of the test partition to `data/processed/test.csv`. The test partition remains strictly held out for final model evaluation.

Both files are processed using the reusable `export_or_reuse_dataset` logic, ensuring that existing valid files are reused without rewriting and missing files are safely created and verified.

In [13]:
# Export processed datasets regenerated from fresh raw splits
df_raw_test_fresh = pd.read_csv(EXPECTED_EXPORT_PATHS["raw_test"])
processed_export_reports = [
    export_dataset(EXPECTED_EXPORT_PATHS["processed_train_val"], df_combined, "Processed Combined (Train + Val)", force_rebuild=FORCE_REBUILD),
    export_dataset(EXPECTED_EXPORT_PATHS["processed_test"], df_raw_test_fresh, "Processed Test", force_rebuild=FORCE_REBUILD),
]

display(pd.DataFrame(processed_export_reports))


,Dataset,Path,Rows,Action
0,Processed Combined (Train + Val),data/processed/train_val.csv,206810,Exported fresh dataset and verified
1,Processed Test,data/processed/test.csv,1162,Exported fresh dataset and verified


## 9. Verify Exported Datasets

Reload each expected CSV file from disk to verify schema fidelity, record count accuracy, serialization consistency, test set isolation, and concatenation order across all columns and rows.

In [14]:
verification_targets = [
    ("Raw Train", EXPECTED_EXPORT_PATHS["raw_train"], df_train),
    ("Raw Validation", EXPECTED_EXPORT_PATHS["raw_val"], df_val),
    ("Raw Test", EXPECTED_EXPORT_PATHS["raw_test"], df_test),
    ("Processed Combined", EXPECTED_EXPORT_PATHS["processed_train_val"], df_combined),
    ("Processed Test", EXPECTED_EXPORT_PATHS["processed_test"], df_test),
]

audit_results = []
for name, file_path, df_source in verification_targets:
    if not file_path.exists():
        raise FileNotFoundError(f"Verification error: Expected file not found: {file_path}")
    
    reloaded_df = pd.read_csv(file_path)
    
    # Verify row count
    if len(reloaded_df) != len(df_source):
        raise ValueError(
            f"Row count mismatch for {name}: expected {len(df_source)}, got {len(reloaded_df)}"
        )
    
    # Verify column structure and ordering
    if list(reloaded_df.columns) != list(df_source.columns):
        raise ValueError(
            f"Column mismatch for {name}: expected {list(df_source.columns)}, got {list(reloaded_df.columns)}"
        )
    
    # Verify all columns and records (accounting for CSV string serialization)
    for col in df_source.columns:
        src_series = df_source[col].fillna("").astype(str).reset_index(drop=True)
        rel_series = reloaded_df[col].fillna("").astype(str).reset_index(drop=True)
        if not src_series.equals(rel_series):
            raise ValueError(f"Value content mismatch in {name} for column '{col}'")
    
    null_count = int(reloaded_df.isnull().sum().sum())
    dup_count = int(reloaded_df.duplicated().sum())
    
    audit_results.append({
        "Dataset": name,
        "File Path": str(file_path.relative_to(REPO_ROOT)),
        "Row Count": len(reloaded_df),
        "Columns": len(reloaded_df.columns),
        "Missing Values": null_count,
        "Duplicate Rows": dup_count,
        "Verified": True,
    })

display(pd.DataFrame(audit_results))

,Dataset,File Path,Row Count,Columns,Missing Values,Duplicate Rows,Verified
0,Raw Train,data/raw/train.csv,205734,4,0,0,True
1,Raw Validation,data/raw/validation.csv,1076,4,0,0,True
2,Raw Test,data/raw/test.csv,1162,4,0,0,True
3,Processed Combined,data/processed/train_val.csv,206810,4,0,0,True
4,Processed Test,data/processed/test.csv,1162,4,0,0,True


### Cross-Split Parity & Partition Alignment Assertions
Verify that raw test identically matches processed test, and processed combined cleanly partitions into train and validation.

In [15]:
# Verify that data/raw/test.csv exactly matches data/processed/test.csv across all columns and records
df_raw_test_check = pd.read_csv(EXPECTED_EXPORT_PATHS["raw_test"])
df_proc_test_check = pd.read_csv(EXPECTED_EXPORT_PATHS["processed_test"])
assert len(df_raw_test_check) == len(df_proc_test_check), "Row count mismatch between raw and processed test"
assert list(df_raw_test_check.columns) == list(df_proc_test_check.columns), "Column mismatch between raw and processed test"
for col in df_raw_test_check.columns:
    raw_s = df_raw_test_check[col].fillna("").astype(str)
    proc_s = df_proc_test_check[col].fillna("").astype(str)
    assert raw_s.equals(proc_s), f"Column '{col}' mismatch between raw test and processed test"

# Verify that data/processed/train_val.csv contains original training records followed by original validation records across all columns
df_proc_combined_check = pd.read_csv(EXPECTED_EXPORT_PATHS["processed_train_val"])
train_len = len(df_train)
val_len = len(df_val)
assert len(df_proc_combined_check) == train_len + val_len, "Combined row count mismatch"
assert list(df_proc_combined_check.columns) == list(df_train.columns), "Combined column mismatch"

# Check all columns and records for training partition (prefix)
df_c_train = df_proc_combined_check.iloc[:train_len].reset_index(drop=True)
for col in df_train.columns:
    expected_s = df_train[col].fillna("").astype(str).reset_index(drop=True)
    actual_s = df_c_train[col].fillna("").astype(str).reset_index(drop=True)
    assert actual_s.equals(expected_s), f"Column '{col}' mismatch in combined training records"

# Check all columns and records for validation partition (suffix)
df_c_val = df_proc_combined_check.iloc[train_len:train_len + val_len].reset_index(drop=True)
for col in df_val.columns:
    expected_s = df_val[col].fillna("").astype(str).reset_index(drop=True)
    actual_s = df_c_val[col].fillna("").astype(str).reset_index(drop=True)
    assert actual_s.equals(expected_s), f"Column '{col}' mismatch in combined validation records"

print("\nIntegrity verification passed: All 5 exported datasets match source data across all columns and records with complete fidelity.")


Integrity verification passed: All 5 exported datasets match source data across all columns and records with complete fidelity.


## 10. Summary

### Dataset Characteristics
- **Benchmark**: Corr2Cause (`causal-nlp/corr2cause`) ingested with verified splits: `train` (205,734 records), `validation` (1,076 records), and `test` (1,162 records).
- **Schema & Data Quality**: Four official features (`input`, `label`, `num_variables`, `template`). All four columns, values, and records are preserved. 0 missing values and 0 duplicate rows detected across all splits.
- **Label Distribution**: Severe class imbalance across all splits (~81.5% Label 0 / refuted causal hypotheses vs. ~18.5% Label 1 / entailed causal hypotheses in train; 87.1% vs. 12.9% in validation; 84.5% vs. 15.5% in test).
- **Target Export Files**:
  - `data/raw/train.csv` (205,734 rows)
  - `data/raw/validation.csv` (1,076 rows)
  - `data/raw/test.csv` (1,162 rows)
  - `data/processed/train_val.csv` (206,810 rows)
  - `data/processed/test.csv` (1,162 rows)
- **Safe Export & Conflict Protocol**: The pipeline conducts a full preflight check across all five destinations before any write. Existing valid files are reused without rewriting; conflicting files immediately halt execution; and missing files are created and verified.
- **Evaluation Protocol**: The test set is held out strictly for final integration and model evaluation. It is never utilized for feature engineering, model architecture exploration, or hyperparameter selection.
- **Pipeline Next Step**: With dataset properties and schemas verified, proceed to downstream feature engineering, entity extraction, and modeling pipelines.